<a href="https://colab.research.google.com/github/likhith040/car-ml/blob/main/car_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -O car.data https://raw.githubusercontent.com/vincm1/UCI_car_data/HEAD/car.data

--2026-10-08 11:43:21--  https://raw.githubusercontent.com/vincm1/UCI_car_data/HEAD/car.data
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51867 (51K) [text/plain]
Saving to: ‘car.data’

car.data            100%[===================>]  50.65K  --.-KB/s    in 0.01s   

2026-10-08 11:43:21 (4.40 MB/s) - ‘car.data’ saved [51867/51867]



In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, silhouette_score, adjusted_rand_score, normalized_mutual_info_score
warnings.filterwarnings("ignore")

cols=['buying','maint','doors','persons','lug_boot','safety','class']
df=pd.read_csv('car.data',header=None,names=cols)
X_cat, y = df[cols[:-1]], df['class']
X = OrdinalEncoder().fit_transform(X_cat)          # for tree models
Xoh = OneHotEncoder().fit_transform(X_cat).toarray()  # for clustering

def models(seed):
    return {
     'Decision Tree (depth 5)': DecisionTreeClassifier(max_depth=5, random_state=seed),
     'Bagging (50 trees, 100 samples)': BaggingClassifier(DecisionTreeClassifier(max_depth=5), n_estimators=50, max_samples=100, random_state=seed),
     'Random Forest (50 trees, 100 samples, depth 5)': RandomForestClassifier(n_estimators=50, max_samples=100, max_depth=5, random_state=seed),
     'AdaBoost (50 stumps-depth2)': AdaBoostClassifier(DecisionTreeClassifier(max_depth=2), n_estimators=50, random_state=seed),
     'Gradient Boosting (100, depth 3)': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=seed)}

print("=== CLASSIFICATION & BOOSTING (50/50 split -> 864 test, as in the notes) ===")
seed=0
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.5,random_state=seed,stratify=y)
res={}
for n,m in models(seed).items():
    m.fit(Xtr,ytr); p=m.predict(Xte); res[n]=(p,accuracy_score(yte,p))
    print(f"{n:50s} {int((p==yte).sum())}/{len(yte)}  acc={res[n][1]*100:.2f}%")
best=max(res,key=lambda k:res[k][1]); print("\nBest:",best)
print(classification_report(yte,res[best][0],zero_division=0))
labs=sorted(y.unique()); print("Confusion matrix (rows=true, cols=pred)",labs); print(confusion_matrix(yte,res[best][0],labels=labs))

print("\n=== Robustness: mean±std accuracy over 10 random 50/50 splits ===")
acc={n:[] for n in models(0)}
for s in range(10):
    a,b,c,d=train_test_split(X,y,test_size=0.5,random_state=s,stratify=y)
    for n,m in models(s).items(): acc[n].append(accuracy_score(d,m.fit(a,c).predict(b)))
for n,v in acc.items(): print(f"{n:50s} {np.mean(v)*100:.2f}% ± {np.std(v)*100:.2f}")

print("\n=== CLUSTERING (k=4, one-hot features; labels used only for evaluation) ===")
print("k  inertia   silhouette")
for k in range(2,9):
    km=KMeans(k,n_init=10,random_state=0).fit(Xoh); print(f"{k}  {km.inertia_:8.1f}  {silhouette_score(Xoh,km.labels_):.3f}")
def purity(lab):
    return sum(pd.Series(y[lab==c]).value_counts().iloc[0] for c in np.unique(lab))/len(y)
km=KMeans(4,n_init=10,random_state=0).fit(Xoh)
print(f"\nSklearn K-Means k=4: silhouette={silhouette_score(Xoh,km.labels_):.3f} purity={purity(km.labels_):.3f} ARI={adjusted_rand_score(y,km.labels_):.3f} NMI={normalized_mutual_info_score(y,km.labels_):.3f}")

# K-Means Neural Network: competitive learning, winner-takes-all, unit-length weights (from your notes)
def kmeans_nn(Xd,k=4,eta=0.05,epochs=30,seed=0,normalise=True):
    r=np.random.RandomState(seed); Xn=Xd/np.linalg.norm(Xd,axis=1,keepdims=True)
    W=Xn[r.choice(len(Xn),k,replace=False)].copy()
    if normalise: W/=np.linalg.norm(W,axis=1,keepdims=True)
    for e in range(epochs):
        for i in r.permutation(len(Xn)):
            j=np.argmax(W@Xn[i])            # activation h_i = w_i . x ; highest wins
            W[j]+=eta*Xn[i]                 # delta w = eta * x  (winner only)
            if normalise: W[j]/=np.linalg.norm(W[j])
    return np.argmax(Xn@W.T,axis=1)
for norm in (True,False):
    lab=kmeans_nn(Xoh,normalise=norm)
    sizes=np.bincount(lab,minlength=4).tolist()
    if len(np.unique(lab))<2:
        print(f"K-Means NN (weights normalised={norm}): cluster sizes={sizes} -> COLLAPSED: one neuron wins every input")
    else:
        print(f"K-Means NN (weights normalised={norm}): cluster sizes={sizes} silhouette={silhouette_score(Xoh,lab):.3f} purity={purity(lab):.3f} ARI={adjusted_rand_score(y,lab):.3f}")

=== CLASSIFICATION & BOOSTING (50/50 split -> 864 test, as in the notes) ===
Decision Tree (depth 5)                            728/864  acc=84.26%
Bagging (50 trees, 100 samples)                    721/864  acc=83.45%
Random Forest (50 trees, 100 samples, depth 5)     692/864  acc=80.09%
AdaBoost (50 stumps-depth2)                        781/864  acc=90.39%
Gradient Boosting (100, depth 3)                   848/864  acc=98.15%

Best: Gradient Boosting (100, depth 3)
              precision    recall  f1-score   support

         acc       0.94      0.98      0.96       192
        good       0.97      0.94      0.96        35
       unacc       1.00      0.98      0.99       605
       vgood       0.97      0.97      0.97        32

    accuracy                           0.98       864
   macro avg       0.97      0.97      0.97       864
weighted avg       0.98      0.98      0.98       864

Confusion matrix (rows=true, cols=pred) ['acc', 'good', 'unacc', 'vgood']
[[189   1   1   1]
